# Anwenderwebsite lokal bauen

Dieses How-to baut dieselben Lehrgrafiken, Downloads und Notebook-Lesefassungen wie Quarto. Die realen Beispiele werden aus `website/_examples.yml` ausgewählt. Voraussetzung: `python -m pip install -e ".[publication]"` im Repository, Quarto für die HTML-Seiten. Für die vorhandenen synthetischen Grafikexporte zusätzlich Chrome/Kaleido.

Die Website läuft statisch, beispielsweise auf GitHub Pages. Sie nutzt das bestehende Repository und Python-Paket beim Build; der Browser führt keine Python- oder Excel-Simulation aus. Eine Veröffentlichung ist ein separater Schritt.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/website_assets.py").exists()
sys.path.insert(0, str(ROOT / "scripts"))
from website_assets import build
output = build(ROOT / "notebooks/local_exports/website_tutorial")
assert (output / "assets/teaching/index.html").exists()
assert (output / "assets/components/index.html").exists()
print("Lehrmodule, öffentliche Unterlagen und Notebook-Lesefassungen erstellt.")

## Lehrgrafiken ausprobieren

Im ersten Modul: Bauteil auswählen, Hülle aktivieren, gekühlte/ungekühlte Zone vergleichen, einzelne Wärmeflüsse ausblenden, aktuellen Zustand als SVG speichern. Der zweite Schnitt zeigt eine ausdrücklich angenommene unbeheizte Kellerzone. Die Kategorienamen stammen aus dem bestehenden Schema; nicht vorhandene Elemente erhalten keine Ersatzzuordnung.

In [ ]:
from IPython.display import HTML, display
from component_teaching import categories
assert len(categories()) == 20
display(HTML('<iframe title="Bauteilkatalog" style="width:100%;height:850px" srcdoc="' + __import__('html').escape((output / "assets/components/index.html").read_text(encoding="utf-8"), quote=True) + '"></iframe>'))

## Quarto rendern und prüfen

Im Repository `quarto render website` ausführen. Der aktive Pre-Render-Schritt
`website_assets.py` erstellt Downloads, Notebook-Lesefassungen, den Explorer und
die konfigurierten Ergebnisgrafiken mit den bestehenden Implementierungsfunktionen.
Quarto rendert anschließend die deutschen QMD-Seiten und das Bewertungstutorial.
Die Ausgabe liegt in `website/_site`.

Englisch ist ein getrenntes Unterprojekt. Sein automatischer Post-Render-Aufruf
ist deaktiviert. Bei einer späteren Veröffentlichung die bestehende englische
Ausgabe bewusst erhalten oder separat bauen; ein deutscher Build allein ist kein
vollständiges zweisprachiges Veröffentlichungspaket.

Unter Windows kann für Kaleido ein vorhandener kompatibler Browser gewählt werden.
Falls Chrome beim Grafikexport hängt, vor dem Build in derselben PowerShell-Sitzung
beispielsweise `$env:BROWSER_PATH = 'C:/Program Files (x86)/Microsoft/Edge/Application/msedge.exe'`
setzen, sofern dieser Pfad vorhanden ist. Das ist eine lokale Laufzeitwahl;
die Website und ihre Ergebnisse ändern sich dadurch nicht.


## Konfigurierte Ergebnisse erzeugen

Website und Publikations-Notebook verwenden denselben Adapter. Hier werden die interaktiven HTML-Ergebnisse ohne erneute statische Bildausgabe erzeugt.

In [ ]:
from website_examples import build as build_examples
reports, checks = build_examples(output / "assets/results", formats=("html",))
assert len(reports) == 4
checks

## Excel-Dokumentation und Abbildungen

Die deutsche Navigation trennt Starten, Modellieren, Analysieren, Deklarieren,
Python & Automatisierung sowie Referenzen. `systems.qmd` und `mobility.qmd`
beschreiben Eingaben; `flexibility.qmd` und `mobility-analysis.qmd` erklären deren
Auswertung. `results.qmd` führt Kennzahlen und Bilanzgrenzen zusammen.

`excel-navigation.qmd` ist die maßgebliche Liste für Blattmarken und Namen.
Beispiel: `Blatt [🔥H]{.excel-menu .darkred} ("Heizung")`.
Felder tragen `.excel-field`, Aktionen `.excel-action`, Eingabeabschnitte
`.excel-section`. Keine zusätzliche Mapping-Datei pflegen.

Schulungsbilder werden durch `website_assets.build` unverändert aus
`docs/website/` nach `media/manual/` kopiert. Weitere verwendete Grafiken liegen
unter `website/illustrations/` und werden direkt von Quarto übernommen.
Die dortige README dokumentiert Herkunft und Reproduktion der Paperausschnitte.
Ein normaler Build benötigt die lokalen Zotero-PDFs nicht.

Für neue Bedienannotationen eine Blattübersicht und lesbare Detailansichten mit
Feldnamen, Einheiten und Zeilen-/Spaltenköpfen verwenden. Die Arbeitsmappenversion
intern dokumentieren und Feldbezeichnungen mit dem vorhandenen Schema abgleichen.
Bild und Erklärungstabelle gemeinsam aktualisieren; Beispielwerte als solche kennzeichnen.


In [ ]:
from website_assets import MANUAL_FIGURES
for name in MANUAL_FIGURES:
    source = ROOT / "docs/website" / f"{name}.png"
    published = output / "media/manual" / source.name
    assert published.read_bytes() == source.read_bytes()
print(f"{len(MANUAL_FIGURES)} Originalabbildungen unverändert veröffentlicht.")


### Visuelle Abnahme

Nach `quarto render website` die Seiten Presets, Blattnavigator und Excel-Dashboard auf Desktop und bei 390 px Breite prüfen. Blattmarken und Pfade müssen umbrechen; Bilder sind auf weißem Hintergrund und per Lightbox vergrößerbar. Fehlende Ansichten sind als „In Vorbereitung“ gekennzeichnet. Linkziele und Anker einschließlich F7 unter `examples.qmd#aspern-f7` prüfen.


## Englisches Unterprojekt

Die englischen Quellen liegen unter `website/en/` und bleiben von der deutschen
Redaktion unberührt. Bei Bedarf separat mit `python scripts/render_english_site.py`
bauen. Das ist für die deutsche Vorschau nicht erforderlich.


## Quellenstand und reproduzierbarer Build

`master` enthält die bearbeitbaren QMD-Seiten, beide Quarto-Konfigurationen, CSS, Python-Skripte, Notebooks und die benötigten Eingabedateien. `gh-pages` enthält ausschließlich die Veröffentlichung einschließlich der vorgesehenen Notebook-Downloads. Generierte Verzeichnisse unter `website/` bleiben ignoriert.

Vor Änderungen lokale, unversionierte und benötigte ignorierte Dateien separat sichern; ein Git-Bundle enthält nur Git-Objekte und Referenzen. Änderungen zunächst auf einem Entwicklungsbranch prüfen. Commit, Übernahme in `master`, Push und Veröffentlichung erfolgen nur nach der jeweiligen Freigabe.

Für einen frischen Build müssen insbesondere die drei `data/exports/Bsp_San_6G*.xlsx`, die Schulungs-PDF und -PPTX, `docs/ComponentTypes.png`, `docs/mobilität.png`, `docs/website/*.png` und `website/illustrations/peq-stufen.png` vorhanden sein. Die eng begrenzten Ignore-Ausnahmen erhalten diese Quellen. `website/_examples.yml` und `scripts/website_assets.py` definieren die verwendeten Daten und Downloads.

In einer frischen Arbeitskopie die bestehenden Abhängigkeiten mit `python -m pip install -e ".[publication]"` bereitstellen; außerdem Quarto und Chrome/Chromium für Kaleido. Anschließend `quarto render website` ausführen. Der Pre-render-Schritt verwendet `website_assets.py`; dadurch bleiben auch die bereits veröffentlichten synthetischen Grafik-URLs aus einem frischen Quellenstand erreichbar. Der Sprachwechsel verweist auf `en/index.html`, da Englisch ein eigenes Quarto-Projekt ist; so wird dessen QMD-Quelle nicht als Ressource kopiert. Die folgende Zelle führt genau diesen Build aus und prüft Seitenumfang, Sprachwechsel und die Trennung von Quellen und Ausgabe. Sie veröffentlicht nichts.


In [ ]:
import subprocess
subprocess.run(["quarto", "render", "website"], cwd=ROOT, check=True)
site = ROOT / "website/_site"
for source in (ROOT / "website").glob("*.qmd"):
    assert (site / source.with_suffix(".html").name).is_file(), source.name
for name in ("mobprofiles.png", "userflexmodel.svg", "flexibility-study.png", "mobility-results.png"):
    assert (site / "illustrations" / name).is_file(), name
print("Deutsche Seiten und Analyse-Abbildungen vorhanden; keine Veröffentlichung.")


## Lokale Vorschau und spätere Veröffentlichung

Nach erfolgreichem Build `quarto preview website --no-browser --no-render` starten.
Die ausgegebene lokale URL öffnen. Startseite, Blattnavigator, Ergebnisse,
Flexibilität und Mobilitätsverhalten auf Desktop und bei 390 px Breite prüfen.
Abbildungen vergrößern, Navigation und Downloads anklicken.

Die folgende Zelle veröffentlicht standardmäßig nichts. Vor einem späteren
GitHub-Pages-Deployment die Vorschau beenden und die Sprachfassungen des fertigen
Ausgabeverzeichnisses prüfen. GitHub Pages verwendet `gh-pages` und den Ordner `/`.


In [ ]:
import subprocess
PUBLISH = False
command = ["quarto", "publish", "gh-pages", "website", "--no-render", "--no-prompt", "--no-browser"]
if PUBLISH:
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Nach erfolgreichem Rendern: " + " ".join(command))


## Interaktive Einbettungen vorübergehend deaktiviert

Die deutschen und englischen Seiten `examples`, `envelope` und `components` laden derzeit keine interaktiven Grafik-Iframes. Die ursprünglichen Einbettungen sind im QMD auskommentiert. Auf der Beispielseite ersetzen die vorhandenen SVG-Exporte die vier Jahresbilanzen und die saisonalen F7-Profile. Der vollständige Stundenverlauf und die Schnittansichten bleiben über separate Links erreichbar und laden erst beim Öffnen. Es werden keine zusätzlichen Grafikdateien oder Datenformate gepflegt.

Für die Prüfung den bestehenden Build `quarto render website` ausführen. Im gerenderten Inhalt dieser sechs Seiten dürfen keine `iframe.pexl-chart` oder `iframe.peq-lesson` vorhanden sein; die SVG-Dateien und Links müssen erreichbar sein. Zur Reaktivierung die betreffenden QMD-Kommentare entfernen und die statischen Ersatzabbildungen entfernen.


## Das Bewertungs-Notebook als Quarto-Seite

`notebooks/howto_assess.ipynb` ist die einzige bearbeitbare Inhaltsquelle des Bewertungstutorials. Texte und Python-Code werden dort geändert. Nach Code- oder Datenänderungen das Notebook ausführen und mit seinen Ausgaben speichern.

Der vorhandene Pre-render-Schritt `scripts/website_assets.py` ruft `prepare_assessment()` auf. Diese Funktion erzeugt `website/assessment.ipynb` mit Seitentitel, Downloadlink und den gespeicherten Ausgaben. Die Kopie ist ignoriert und wird bei jedem Build überschrieben; sie wird nicht manuell gepflegt. Der Zweck dieser einen abgeleiteten Datei ist, das außerhalb des Website-Projekts liegende Notebook mit dessen Navigation und Gestaltung nativ zu rendern. Es gibt keine zweite Text- oder Berechnungsquelle und kein neues Datenschema. Der Download wird wie die übrigen Downloads ohne gespeicherte Ausgaben erzeugt.

In `website/_quarto.yml` steht unter `project.render` zusätzlich `assessment.ipynb`. Derselbe Dateiname steht in der Seitenleiste unter „Python & Automatisierung“. Quarto rendert Markdown, aufklappbaren Python-Code, Tabellen und interaktive Plotly-Ausgaben gemeinsam nach `website/_site/assessment.html`. `execute.enabled: false` in der erzeugten Kopie verhindert eine erneute Ausführung beim Rendern. Die öffentliche Seite benötigt keinen Python-Server.

Die folgende Zelle zeigt genau diesen Vorbereitungsschritt. Danach baut `quarto render website` die Website wie gewohnt. Auf „Python-Schnittstelle und Notebooks“ führt ein Link zum Tutorial; die Seite selbst bietet den Notebook-Download. Es wird nichts veröffentlicht.


In [ ]:
from website_assets import prepare_assessment
assessment_page = prepare_assessment(output)
assessment = __import__("nbformat").read(assessment_page, as_version=4)
print("Quarto-Eingabe:", assessment_page)
print("Zellen einschließlich Seitenkopf:", len(assessment.cells))
print("Download:", output / "downloads/howto_assess.ipynb")

## Vom Projektexport zu beiden Bewertungsseiten

Die Website bietet zwei ergänzende Ansichten:

- **Technisches Bewertungs-Tutorial:** `howto_assess.ipynb` wird als `assessment.html` mit Texten, Code und gespeicherten Diagrammen dargestellt.
- **Projekt-Varianten-Explorer:** `howto_assessment_explained.ipynb` liefert die fachlichen Texte; `assessment-explained.html` bettet den interaktiven Explorer ohne technischen Begleittext ein. Die vollständige Ansicht liegt unter `assets/assessment-explained/index.html`.

### 1. Projektexport auswählen

Eine vorhandene PEExcel-Exportdatei mit gespeicherten Szenarien verwenden. Der Explorer berechnet keine neuen Varianten. Die Codezelle unten erzeugt eine eigenständige HTML-Datei mit allen auswählbaren Szenarien. Bei eigenen Projekten `QUELLE` ersetzen. Die Vorauswahl nutzt passende Lehrbeispiele, sonst die ersten drei Szenarien. Der tatsächliche Dateiname erscheint als Datengrundlage.

In der Kommandozeile ist derselbe Ablauf möglich:

```powershell
python scripts/assessment_explained.py --source "C:/Projekte/Projektexport.xlsx" --output "notebooks/local_exports/mein-projekt/index.html"
```

Die neue Option `source` verwendet denselben Import und dieselben Plotfunktionen. Es wird keine zusätzliche Projektkonfiguration benötigt. Fachliche Texte werden im erklärenden Notebook bearbeitet; ein eigener Projektexport ersetzt diese Texte nicht automatisch.


In [ ]:
from assessment_explained import build as build_explorer

QUELLE = ROOT / "data/exports/Klimsta_Abschlussprojekt_Varianten_V3.xlsx"
explorer_html = build_explorer(
    ROOT / "notebooks/local_exports/projekt_varianten_explorer/index.html",
    source=QUELLE,
)
assert explorer_html.is_file()
print(explorer_html)

### 2. Technisches Notebook mit demselben Export ausführen

In `notebooks/howto_assess.ipynb` die Variable `QUELLE` auf dieselbe Exportdatei setzen. Das Notebook zeigt zunächst die vorhandenen Varianten. Anschließend `AUSWAHL` an deren tatsächliche Namen anpassen, alle Zellen ausführen und mit Ausgaben speichern. Diese gespeicherten Ausgaben übernimmt Quarto; der Website-Build führt das technische Notebook nicht nochmals aus.

### 3. Beide Ansichten in PEExcel-Pages bauen

```powershell
quarto render website
```

Der vorhandene Pre-render-Schritt `website_assets.py` erzeugt nun auch den Explorer sowie beide Notebook-Downloads. Die Navigation enthält den Explorer unter „Analysieren“; unter „Python-Schnittstelle und Notebooks“ sind beide Ansichten verlinkt.

Der reguläre Website-Build verwendet den mitgelieferten Lehrprojektexport. Soll die Website dauerhaft einen anderen Export zeigen, in `scripts/website_assets.py` beim Aufruf von `build_explorer(...)` zusätzlich `source=ROOT / "data/exports/MeinProjekt.xlsx"` setzen. Die Quelldatei muss beim Build verfügbar sein. Ein zuvor manuell erzeugtes Explorer-HTML wird vom regulären Build ersetzt.

Zu prüfen sind `website/_site/assessment.html`, `website/_site/assessment-explained.html` und `website/_site/assets/assessment-explained/index.html`. Beide Downloads liegen unter `website/_site/downloads/`.

### 4. Nach Prüfung veröffentlichen

```powershell
quarto publish gh-pages website --no-render
```

Dieser Befehl veröffentlicht die fertig gebaute Website. Der Build und die Beispielzelle oben veröffentlichen nichts. Die eigenständige Explorer-HTML-Datei kann ebenso in einen anderen Quarto-Bericht kopiert werden; der konkrete Weg für `peq-higrube` steht in `docs/development/assessment_explained.md`.
